# LittleBit-2 QAT for Qwen3-0.6B

Enable Internet and choose a GPU before training. Edit the settings below, then use **Save Version > Save & Run All**. See `train/README.md` for resume, CPU evaluation and upload.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
TINY = os.getenv("LITTLEBIT_TINY") == "1"
RUN_TRAIN = True
RUN_GGUF = False
UPLOAD = False

TRAINING_COMMIT = ""
RESUME = ""  # e.g. /kaggle/input/notebooks/owner/run/littlebit-train/resume.pt
RESULTS = ""  # use an attached littlebit-train directory for CPU evaluation or upload
HF_REPO = ""  # owner/repository for the optional private upload

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT = Path(os.getenv("LITTLEBIT_OUTPUT", str(WORK / "littlebit-train"))).resolve()
TOKENS = 64 if TINY else 2_097_152
SEQ_LEN = 16 if TINY else 2048
C4_TOKENS = 8_388_608  # 0 tokenizes the complete shard
MINUTES = 630
LOGIT_CHUNK = 128

In [ ]:
LOCAL_REPO = os.getenv("LITTLEBIT_REPO")
SCRATCH = Path("/kaggle/temp")
if LOCAL_REPO:
    repo = Path(LOCAL_REPO).resolve()
    python = sys.executable
else:
    SCRATCH.mkdir(parents=True, exist_ok=True)
    repo = SCRATCH / "littlebit-gguf"
    venv = SCRATCH / "littlebit-env"
    python = str(venv / "bin/python")
    upstream_repo = "https://github.com/GeniusDevam/littlebit-gguf.git"

    prior = (
        Path(RESULTS).resolve()
        if RESULTS
        else Path(RESUME).resolve().parent
        if RESUME
        else None
    )
    revision = TRAINING_COMMIT
    if not revision and prior and (prior / "summary.json").is_file():
        revision = json.loads((prior / "summary.json").read_text())["training_commit"]
    revision = revision or "train-r1"
    if not (repo / ".git").is_dir():
        subprocess.run(["git", "init", "-q", repo], check=True)
    subprocess.run(
        ["git", "fetch", "--depth", "1", upstream_repo, revision], cwd=repo, check=True
    )
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True)

    if not Path(python).is_file():
        subprocess.run(
            [sys.executable, "-m", "venv", "--without-pip", venv], check=True
        )
    pip_install = [sys.executable, "-m", "pip", "--python", python, "install"]
    wheels = "cpu" if TINY or not RUN_TRAIN else "cu126"
    subprocess.run(
        pip_install
        + [
            f"torch==2.14.0+{wheels}",
            "--index-url",
            f"https://download.pytorch.org/whl/{wheels}",
        ],
        check=True,
    )
    subprocess.run(
        pip_install + ["-r", repo / "train/requirements.txt"],
        check=True,
    )

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
print(
    "Source commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)

In [ ]:
if RUN_TRAIN:
    command = [
        python,
        "-u",
        "-m",
        "train.train_qat",
        "--output",
        OUTPUT,
        "--session-start",
        str(SESSION_START),
        "--minutes",
        str(MINUTES),
        "--tokens",
        str(TOKENS),
        "--seq-len",
        str(SEQ_LEN),
        "--c4-tokens",
        str(C4_TOKENS),
        "--logit-chunk",
        str(LOGIT_CHUNK),
    ]
    if RESUME:
        command += ["--resume", RESUME]
    if TINY:
        command += ["--tiny", "--cpu", "--log-every", "1", "--learning-rate", "0.0001"]
    if os.getenv("LITTLEBIT_UPSTREAM"):
        command += ["--upstream", os.environ["LITTLEBIT_UPSTREAM"]]
    subprocess.run(command, cwd=repo, check=True)
    assert (OUTPUT / "littlebit/littlebit_config.json").is_file()
    assert (OUTPUT / "resume.pt").is_file()
    print(json.loads((OUTPUT / "summary.json").read_text()))
else:
    print("Training is off.")

## Optional CPU evaluation

For a separate CPU session, attach the training output, set `RESULTS` to its path, set `RUN_TRAIN = False`, and enable `RUN_GGUF`. The cell builds the patched tools, converts both embedding types and runs perplexity on the same test text for both files and the BF16 teacher. It also records the CPU and a short generation sample.

In [ ]:
if RUN_GGUF:
    results = Path(RESULTS).resolve() if RESULTS else OUTPUT
    subprocess.run(
        [
            python,
            "-u",
            "-m",
            "train.evaluate_gguf",
            results,
            WORK / "littlebit-evaluation",
        ],
        cwd=repo,
        check=True,
    )
else:
    print("GGUF conversion and evaluation are off.")

## Optional private upload

Set `HF_REPO`, add `HF_TOKEN` in Kaggle Secrets and grant this notebook access to it. Enable `UPLOAD` to publish the format-exact checkpoint folder to your Hugging Face repository.

In [ ]:
if UPLOAD:
    if not HF_REPO:
        raise ValueError("Set HF_REPO to owner/repository before enabling upload.")
    from kaggle_secrets import UserSecretsClient

    results = Path(RESULTS).resolve() if RESULTS else OUTPUT
    upload = """
import sys
from huggingface_hub import HfApi

try:
    api = HfApi(token=sys.stdin.read())
    api.create_repo(sys.argv[1], repo_type="model", private=True, exist_ok=True)
    if not api.model_info(sys.argv[1]).private:
        sys.exit("The destination repository must be private.")
    api.upload_folder(repo_id=sys.argv[1], folder_path=sys.argv[2], repo_type="model")
except Exception:
    sys.exit("Upload failed. Check the secret and repository permissions.")
"""
    subprocess.run(
        [python, "-c", upload, HF_REPO, results / "littlebit"],
        input=UserSecretsClient().get_secret("HF_TOKEN"),
        text=True,
        check=True,
    )
else:
    print("Upload is off.")